# Dealing with Counts: Binarization

_Feature Engineering (Zheng & Casari)_

**Sometimes only presence matters, not how much — clip a raw count to 0/1.**

A count is a non-negative integer: how many times something happened. Counts are everywhere — plays, clicks, words, visits.

       The trap: counts can be wildly skewed. A few items get astronomically high counts; most get tiny ones. Plotted, the distribution has a long right tail. Fed to a model, the giant values dominate.

---

This notebook is a practice scaffold for the **Dealing with Counts: Binarization** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — pandas

In [ ]:
import pandas as pd
import numpy as np

# Echo Nest taste profile subset (Million Song Dataset).
# Columns: user (hash), song (hash), listen_count (int).
# Download: github.com/alicezheng/feature-engineering-book  (and the MSD/Echo Nest site)
listen_count = pd.read_csv(
    'train_triplets.txt', sep='\t', header=None,
    names=['user', 'song', 'listen_count']
)

# ----- the raw counts are heavily skewed -----
print(listen_count['listen_count'].describe())
# one user can play a single song hundreds of times, while most plays are 1-2:
#   mean  ~ 3, max in the hundreds -> a long right tail

# ----- BINARIZE: for a recommender we only care WHETHER a user listened -----
# clip every positive play count to 1; absence stays 0.
listen_count['listen'] = (listen_count['listen_count'] >= 1).astype(int)

# equivalently, with scikit-learn's Binarizer (threshold = 0 keeps >0 as 1):
# from sklearn.preprocessing import binarize
# listen_count['listen'] = binarize(listen_count[['listen_count']], threshold=0.0)

print(listen_count[['listen_count', 'listen']].head(10))
#    listen_count  listen
# 0             1       1
# 1             2       1
# 2             1       1
# 3           312       1     <- the super-fan's count collapses to 1
# 4             1       1

# The 'listen' column is the robust presence/absence signal the book feeds
# to the recommender, instead of the outlier-prone raw 'listen_count'.

## Visualize the data & results

_On a real right-skewed count-like feature, what does the raw distribution look like, and how does binarizing at the median split the classes?_

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()                 # 569 real tumor records
names = list(data.feature_names)
X, y = data.data, data.target               # label: 1 = benign, 0 = malignant

# pick a real count-like, right-skewed feature: 'mean area'
col = X[:, names.index('mean area')]
print("min/median/max =", round(col.min(),1), round(np.median(col),1), round(col.max(),1))
# -> 143.5 / 551.1 / 2501.0  (a long right tail)
skew = ((col - col.mean())**3).mean() / col.std()**3
print("skew =", round(float(skew), 2))      # -> 1.64, heavily right-skewed

# raw distribution: histogram counts pile up on the left
counts, edges = np.histogram(col, bins=8)
print("hist counts =", counts.tolist())     # -> [164, 249, 69, 61, 14, 8, 1, 3]

# BINARIZE at the median (no natural zero, so split the rows in half)
thr = np.median(col)
b = (col >= thr).astype(int)
print("n(0), n(1) =", int((b == 0).sum()), int((b == 1).sum()))   # -> 284, 285

# is the 0/1 split informative? class balance within each half
for v in (0, 1):
    sel = b == v
    print("bin", v, "-> P(benign) =", round(float(y[sel].mean()), 3))
# bin 0 -> 0.940   (small-area half is mostly benign)
# bin 1 -> 0.316   (large-area half is mostly malignant)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You have a "listen_count" column from the Echo Nest data: most values are 1 or 2, but one user played a song 980 times. You are building a song recommender. Should you feed the raw count, and what does binarizing change?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Ask what the recommender actually needs from this column. — _The training signal is "this user likes this song." Presence of an interaction is the weak positive; the exact replay count is mostly noise._
- Notice the 980 is a power-user artifact. — _A heavy-tailed count lets one row dominate a linear model's loss and distort coefficients — the value is about that user's habit, not preference strength._
- Apply b = 1[x >= 1] to collapse every positive count to 1. — _Binarizing maps 1, 2, and 980 all to 1, removing the skew and the outlier's leverage. The feature becomes "listened at all"._

**Answer:** Don't feed the raw count: it is heavy-tailed and the 980 would dominate. Binarize with threshold 1 — the column becomes a 0/1 "did the user listen at all" flag, which is the robust preference signal the book recommends for an implicit-feedback recommender.

</details>

**Problem 2.** A numeric feature "session_length" has no natural zero event and you still want a single binary split. How do you pick the threshold, and what is the trade-off?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use the median of the feature as the threshold t. — _With no natural "did it happen" cutoff, the median splits the rows into a low half and a high half, so neither bin is empty._
- Apply b = 1[x >= median] to every row. — _This produces a balanced 0/1 feature: roughly half the rows are 1 and half are 0._
- Accept that you have discarded the magnitude. — _Binarizing throws away how far above or below the median each value is. If that magnitude is signal, prefer a log transform or binning instead._

**Answer:** Threshold at the median, giving a balanced 0/1 split (b = 1[x >= median]). The trade-off is that you discard all magnitude information — if the magnitude matters, use a log/power transform or binning rather than binarizing.

</details>